In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from joblib import dump

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.features.tfidf import create_tfidf_vectorizer
from src.models.train import create_models
from src.models.evaluate import evaluate_model

In [3]:
train_df = pd.read_csv(
    "../data/processed/train_processed.csv"
)

test_df = pd.read_csv(
    "../data/processed/test_processed.csv"
)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (120000, 9)
Test shape : (7600, 6)


In [4]:
print(train_df.columns.tolist())

['label', 'title', 'description', 'category', 'text', 'word_count', 'clean_text', 'original_word_count', 'clean_word_count']


In [5]:
print(train_df[["clean_text", "label"]].head())

                                          clean_text  label
0  wall st bear claw back black reuters reuters s...      3
1  carlyle look toward commercial aerospace reute...      3
2  oil economy cloud stock outlook reuters reuter...      3
3  iraq halt oil export main southern pipeline re...      3
4  oil price soar alltime record posing new menac...      3


In [6]:
#Prepare X and y
X_train_text = train_df["clean_text"].fillna("")

X_test_text = test_df["clean_text"].fillna("")

y_train = train_df["label"]

y_test = test_df["label"]

In [7]:
print("Training documents:", len(X_train_text))
print("Testing documents :", len(X_test_text))

Training documents: 120000
Testing documents : 7600


In [8]:
#Create TF-IDF
tfidf = create_tfidf_vectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

In [9]:
#Fit ONLY on training data
X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

In [10]:
#
X_test_tfidf = tfidf.transform(
    X_test_text
)

In [11]:
#Prepare X and y
X_train_text = train_df["clean_text"].fillna("")

X_test_text = test_df["clean_text"].fillna("")

y_train = train_df["label"]

y_test = test_df["label"]

In [12]:
#Create TF-IDF
tfidf = create_tfidf_vectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

In [13]:
#Fit ONLY on training data
X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

In [14]:
X_test_tfidf = tfidf.transform(
    X_test_text
)

In [15]:
X_test_tfidf = tfidf.fit_transform(X_test_text)

In [16]:
print("X_train shape:", X_train_tfidf.shape)
print("X_test shape :", X_test_tfidf.shape)

X_train shape: (120000, 50000)
X_test shape : (7600, 25386)


In [17]:
#Inspect the vocabulary
feature_names = tfidf.get_feature_names_out()

print("Number of features:", len(feature_names))

print("\nFirst 50 features:")
print(feature_names[:50])

Number of features: 25386

First 50 features:
['aa' 'aapl' 'aaron' 'ab' 'ababa' 'ababa reuters' 'abandon' 'abandoned'
 'abarrel' 'abb' 'abbas' 'abbey' 'abbey national' 'abbey takeover'
 'abbott' 'abc' 'abdication' 'abdication father' 'abducted'
 'abducted lebaneseamerican' 'abduction' 'abductor' 'abdullah'
 'aberration' 'abide' 'abidjan' 'abidjan ivory' 'abidjan reuters'
 'ability' 'ability run' 'abimael' 'abimael guzman' 'able' 'able play'
 'able use' 'abnormality' 'aboard' 'aboard international' 'aboriginal'
 'aborted' 'abound' 'abraham' 'abroad' 'abruptly' 'abruptly resigned'
 'absence' 'absolutely' 'absorb' 'absorbing' 'absorbs']


In [18]:
#Train the three models
models = create_models()

In [19]:
print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape :", X_test_tfidf.shape)

X_train_tfidf shape: (120000, 50000)
X_test_tfidf shape : (7600, 25386)


In [20]:
from src.features.tfidf import create_tfidf_vectorizer

tfidf = create_tfidf_vectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# IMPORTANT:
# Fit ONLY on training data
X_train_tfidf = tfidf.fit_transform(X_train_text)

# Transform test using the SAME vectorizer
X_test_tfidf = tfidf.transform(X_test_text)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape :", X_test_tfidf.shape)

X_train_tfidf shape: (120000, 50000)
X_test_tfidf shape : (7600, 50000)


In [21]:
#Then run your model-training cell
models = create_models()

results = []
trained_models = {}

for model_name, model in models.items():

    print(f"\nTraining {model_name}...")

    model.fit(X_train_tfidf, y_train)

    trained_models[model_name] = model

    result = evaluate_model(
        model,
        X_test_tfidf,
        y_test,
        model_name
    )

    results.append(result)


Training Naive Bayes...
Naive Bayes
Accuracy : 0.9042
Precision: 0.9040
Recall   : 0.9042
F1 Score : 0.9039

Classification Report:
              precision    recall  f1-score   support

           1       0.92      0.90      0.91      1900
           2       0.95      0.98      0.96      1900
           3       0.88      0.85      0.86      1900
           4       0.87      0.89      0.88      1900

    accuracy                           0.90      7600
   macro avg       0.90      0.90      0.90      7600
weighted avg       0.90      0.90      0.90      7600


Training Logistic Regression...
Logistic Regression
Accuracy : 0.9211
Precision: 0.9209
Recall   : 0.9211
F1 Score : 0.9209

Classification Report:
              precision    recall  f1-score   support

           1       0.94      0.91      0.92      1900
           2       0.96      0.98      0.97      1900
           3       0.89      0.89      0.89      1900
           4       0.90      0.90      0.90      1900

    accurac

In [22]:
results_df = pd.DataFrame(results)

results_df[
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1_score"
    ]
]

,model,accuracy,precision,recall,f1_score
0,Naive Bayes,0.904211,0.903965,0.904211,0.903879
1,Logistic Regression,0.921053,0.920920,0.921053,0.920897
2,Linear SVM,0.923553,0.923494,0.923553,0.923457


In [23]:
print("Train features:", X_train_tfidf.shape[1])
print("Test features :", X_test_tfidf.shape[1])
print("Vectorizer features:", len(tfidf.vocabulary_))

Train features: 50000
Test features : 50000
Vectorizer features: 50000


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

def create_tfidf_vectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
):
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df,
        sublinear_tf=sublinear_tf
    )

    return vectorizer

In [25]:
import pandas as pd

train_df = pd.read_csv("../data/processed/train_processed.csv")
test_df = pd.read_csv("../data/processed/test_processed.csv")

print(train_df.shape)
print(test_df.shape)

(120000, 9)
(7600, 6)


In [26]:
X_train_text = train_df["clean_text"].fillna("")
X_test_text = test_df["clean_text"].fillna("")

y_train = train_df["label"]
y_test = test_df["label"]

print(X_train_text.shape)
print(X_test_text.shape)

(120000,)
(7600,)


In [27]:
#Create ONE TF-IDF vectorizer:
from src.features.tfidf import create_tfidf_vectorizer

tfidf = create_tfidf_vectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

In [28]:
#Fit only on training data:
X_train_tfidf = tfidf.fit_transform(X_train_text)

X_test_tfidf = tfidf.transform(X_test_text)

In [29]:
#Check the dimensions:
print("X_train_tfidf:", X_train_tfidf.shape)
print("X_test_tfidf :", X_test_tfidf.shape)

X_train_tfidf: (120000, 50000)
X_test_tfidf : (7600, 50000)


In [30]:
#Then train your models
from src.models.train import create_models
from src.models.evaluate import evaluate_model

models = create_models()

results = []
trained_models = {}

for model_name, model in models.items():

    print(f"\nTraining {model_name}...")

    model.fit(X_train_tfidf, y_train)

    trained_models[model_name] = model

    result = evaluate_model(
        model,
        X_test_tfidf,
        y_test,
        model_name
    )

    results.append(result)


Training Naive Bayes...
Naive Bayes
Accuracy : 0.9042
Precision: 0.9040
Recall   : 0.9042
F1 Score : 0.9039

Classification Report:
              precision    recall  f1-score   support

           1       0.92      0.90      0.91      1900
           2       0.95      0.98      0.96      1900
           3       0.88      0.85      0.86      1900
           4       0.87      0.89      0.88      1900

    accuracy                           0.90      7600
   macro avg       0.90      0.90      0.90      7600
weighted avg       0.90      0.90      0.90      7600


Training Logistic Regression...
Logistic Regression
Accuracy : 0.9211
Precision: 0.9209
Recall   : 0.9211
F1 Score : 0.9209

Classification Report:
              precision    recall  f1-score   support

           1       0.94      0.91      0.92      1900
           2       0.96      0.98      0.97      1900
           3       0.89      0.89      0.89      1900
           4       0.90      0.90      0.90      1900

    accurac

In [31]:
results_df = pd.DataFrame(results)

results_df[
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1_score"
    ]
]

,model,accuracy,precision,recall,f1_score
0,Naive Bayes,0.904211,0.903965,0.904211,0.903879
1,Logistic Regression,0.921053,0.920920,0.921053,0.920897
2,Linear SVM,0.923553,0.923494,0.923553,0.923457


In [32]:
results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1_score"
    ]
]

In [33]:
results_df.to_csv(
    "../models/tfidf_results.csv",
    index=False
)

print("Phase 3 results saved.")

Phase 3 results saved.
